# Chapter 30: Signals through depth

Checks [chapter 30](../../book/30-signals-through-depth.md). Three stand-in models, each the one a theorem is about.
**Scalar gains and plates** are exact: a gain is a number, a polarizer plate is the projection `light -> inner(axis, light) * axis`
in `R^d` (NumPy, float64), and a bypass plate is `light + plate(light)`. **Sigmoid stacks** are the chain rule on a fine grid.
**ReLU networks without biases** are the model of `he_gain`: `layer_count` layers of width `input_count`, weights
`Normal(0, weight_variance)` with `weight_variance = constant / input_count`, inputs standard normal, seeded. Their gradient is the
backward pass of a random read-out; the chapter does not model the backward mask of a ReLU (see "What is not shown"), so that
measurement is reported as a measurement. These are measurements, not proofs: the proofs are in `VanishingGradients.lean`,
`Gain.lean` and `Activations.lean`. Run every cell, top to bottom. A failed check stops the notebook.

In [1]:
import sys
sys.path.append("../part1")

from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt
from common import use_style, save_figure, report, NEAREST, VERTICAL, BOUND, MODEL, BEFORE

use_style()
SEED = 30


def make_plate(axis):
    """A polarizer: it keeps the part of the light along the axis (the axis has length one)."""
    return np.outer(axis, axis)


def make_bypass_plate(axis):
    """A plate with a bypass: the light plus what the plate passes."""
    return np.eye(len(axis)) + make_plate(axis)


def make_unit_axis(generator, dimension):
    axis = generator.standard_normal(dimension)
    return axis / np.linalg.norm(axis)

## Claim 1: a gain below one dies, and a gain above one grows (`gain_pow_le_exp`, `one_add_mul_le_gain_pow`)

For every gain in `[0, 1]` on a grid of 101 values and every depth from 0 to 300, `gain ^ layer_count <= exp(-layer_count * (1 - gain))`.
For gains in `[1, 3]` and the same depths, `gain ^ layer_count >= 1 + layer_count * (gain - 1)`. Prediction: both hold everywhere, and at gain 0.9
and 50 layers the numbers are 0.005 and 0.0067, and at gain 1.1 they are 117 and 6, as in the chapter. The control breaks the hypothesis
`0 <= gain`: with gain `-2` and 2 layers the signal is `4` and the bound is `exp(-6)`, so the bound must fail there. These numbers come from evaluating the formulas on a grid, so they are **from the formula**, not measured. Any failure of the first two
would count against the claim.

In [2]:
layer_counts = np.arange(0, 301)
dying_gains = np.linspace(0.0, 1.0, 101)
growing_gains = np.linspace(1.0, 3.0, 101)

worst_excess = -np.inf
for gain in dying_gains:
    signal = gain ** layer_counts
    decay_bound = np.exp(-(layer_counts * (1 - gain)))
    worst_excess = max(worst_excess, np.max(signal - decay_bound))
report("(from the formula) gain ^ layer_count <= exp(-layer_count (1 - gain)) for 101 gains in [0, 1], depths 0 to 300", worst_excess <= 1e-15,
       f"largest signal minus bound {worst_excess:.1e}")

worst_shortfall = np.inf
for gain in growing_gains:
    signal = gain ** layer_counts.astype(float)
    growth_bound = 1 + layer_counts * (gain - 1)
    worst_shortfall = min(worst_shortfall, np.min((signal - growth_bound) / signal))
report("(from the formula) gain ^ layer_count >= 1 + layer_count (gain - 1) for 101 gains in [1, 3], depths 0 to 300", worst_shortfall >= -1e-12,
       f"smallest relative margin {worst_shortfall:.1e}")

for gain, layer_count in ((0.9, 50), (1.1, 50)):
    print(f"from the formula: gain {gain}, {layer_count} layers: signal {gain ** layer_count:.4g}, "
          f"decay bound {np.exp(-layer_count * (1 - gain)):.4g}, growth bound {1 + layer_count * (gain - 1):.4g}")

control_signal, control_bound = (-2.0) ** 2, np.exp(-(2 * (1 - -2.0)))
report("(from the formula) control: with gain -2 (hypothesis broken) the decay bound fails at 2 layers", control_signal > control_bound,
       f"signal {control_signal:.1f}, bound {control_bound:.4f}")

[PASS] (from the formula) gain ^ layer_count <= exp(-layer_count (1 - gain)) for 101 gains in [0, 1], depths 0 to 300  (largest signal minus bound 0.0e+00)
[PASS] (from the formula) gain ^ layer_count >= 1 + layer_count (gain - 1) for 101 gains in [1, 3], depths 0 to 300  (smallest relative margin 0.0e+00)
from the formula: gain 0.9, 50 layers: signal 0.005154, decay bound 0.006738, growth bound -4
from the formula: gain 1.1, 50 layers: signal 117.4, decay bound 148.4, growth bound 6
[PASS] (from the formula) control: with gain -2 (hypothesis broken) the decay bound fails at 2 layers  (signal 4.0, bound 0.0025)


True

## Claim 2: a plate never lengthens the light, and a fixed turn dims it (`norm_plate_le`, `plates_vanish`)

Take 1000 random unit axes and random lights in `R^5` and check `||plate(light)|| <= ||light||`. Then take plates at angles
`turn_angle, 2 turn_angle, ...` in the plane, with `turn_angle = 0.3`, and light along the first direction. Prediction: after `layer_count` plates the light has
length exactly `cos(turn_angle) ^ layer_count`, below `exp(-layer_count (1 - cos(turn_angle)))`, for 100 layers. The control uses an axis of length 2
(`||axis|| = 1` broken): then the plate must lengthen some light.

In [3]:
generator = np.random.default_rng(SEED)
DIMENSION = 5
worst_growth = -np.inf
for trial in range(1000):
    axis = make_unit_axis(generator, DIMENSION)
    light = generator.standard_normal(DIMENSION) * generator.uniform(0.1, 10)
    worst_growth = max(worst_growth, np.linalg.norm(make_plate(axis) @ light) / np.linalg.norm(light))
report("a plate never lengthens the light: 1000 random unit axes and lights in R^5", worst_growth <= 1 + 1e-12,
       f"largest length ratio {worst_growth:.6f}")

TURN_ANGLE = 0.3
PLATE_STACK_DEPTH = 100
light = np.array([1.0, 0.0])
lengths = [1.0]
for layer in range(1, PLATE_STACK_DEPTH + 1):
    plate_angle = TURN_ANGLE * layer
    light = make_plate(np.array([np.cos(plate_angle), np.sin(plate_angle)])) @ light
    lengths.append(np.linalg.norm(light))
lengths = np.array(lengths)
plate_layers = np.arange(PLATE_STACK_DEPTH + 1)
predicted_lengths = np.cos(TURN_ANGLE) ** plate_layers
decay_bounds = np.exp(-plate_layers * (1 - np.cos(TURN_ANGLE)))
report("the light after layer_count plates has length cos(turn_angle) ^ layer_count", np.abs(lengths - predicted_lengths).max() < 1e-12,
       f"largest gap {np.abs(lengths - predicted_lengths).max():.1e}")
report("and it is below exp(-layer_count (1 - cos(turn_angle)))", np.all(lengths <= decay_bounds + 1e-15),
       f"length after 100 plates {lengths[-1]:.3e}, bound {decay_bounds[-1]:.3e}")

long_axis = 2.0 * make_unit_axis(generator, DIMENSION)
longest_ratio = 0.0
for trial in range(1000):
    test_light = generator.standard_normal(DIMENSION)
    longest_ratio = max(longest_ratio, np.linalg.norm(make_plate(long_axis) @ test_light) / np.linalg.norm(test_light))
report("control: with an axis of length 2 a plate does lengthen light", longest_ratio > 1.5, f"largest length ratio {longest_ratio:.2f}")

[PASS] a plate never lengthens the light: 1000 random unit axes and lights in R^5  (largest length ratio 0.988989)
[PASS] the light after layer_count plates has length cos(turn_angle) ^ layer_count  (largest gap 3.3e-16)
[PASS] and it is below exp(-layer_count (1 - cos(turn_angle)))  (length after 100 plates 1.037e-02, bound 1.149e-02)
[PASS] control: with an axis of length 2 a plate does lengthen light  (largest length ratio 3.87)


True

## Claim 3: the gradient comes back through the same plates in the opposite order (`inner_plate_comm`, `inner_runPlates`)

Take 8 random unit axes in `R^6`, and 300 random pairs `light`, `other_light`. Prediction: `<plates(light), other_light>` equals `<light, reversed_plates(other_light)>` to rounding
error (about `1e-15`). The control sends `other_light` back through the plates in the **same** order; it must disagree by a visible amount.

In [4]:
generator = np.random.default_rng(SEED + 1)
DIMENSION, PLATE_COUNT = 6, 8
worst_gap = 0.0
worst_gap_same_order = 0.0
for trial in range(300):
    plates = [make_plate(make_unit_axis(generator, DIMENSION)) for _ in range(PLATE_COUNT)]
    light = generator.standard_normal(DIMENSION)
    other_light = generator.standard_normal(DIMENSION)
    forward, backward, same_order = light.copy(), other_light.copy(), other_light.copy()
    for plate in plates:
        forward = plate @ forward
    for plate in reversed(plates):
        backward = plate @ backward
    for plate in plates:
        same_order = plate @ same_order
    forward_reading = forward @ other_light
    worst_gap = max(worst_gap, abs(forward_reading - light @ backward))
    worst_gap_same_order = max(worst_gap_same_order, abs(forward_reading - light @ same_order))
report("<plates(light), other_light> = <light, reversed plates(other_light)>, 300 trials", worst_gap < 1e-12, f"largest gap {worst_gap:.1e}")
report("control: sending it back in the same order does not agree", worst_gap_same_order > 1e-3, f"largest gap {worst_gap_same_order:.3f}")

[PASS] <plates(light), other_light> = <light, reversed plates(other_light)>, 300 trials  (largest gap 1.0e-17)
[PASS] control: sending it back in the same order does not agree  (largest gap 0.047)


True

## Claim 4: plates with a bypass blow up (`le_norm_iterate_skipPlate`)

A bypass plate on one unit axis in `R4`, applied `plate_count` times to random light, for `plate_count` from 0 to 40 and 200 random lights.
Prediction: the overlap with the axis is exactly `2 ^ plate_count` times the first overlap, and the length is at least `2 ^ plate_count * |first overlap|`.
The control is the same plates without the bypass: the overlap must not grow at all.

In [5]:
generator = np.random.default_rng(SEED + 2)
DIMENSION = 4
worst_overlap_gap = 0.0
bound_holds = True
plain_overlap_change = 0.0
for trial in range(200):
    axis = make_unit_axis(generator, DIMENSION)
    first_light = generator.standard_normal(DIMENSION)
    first_overlap = axis @ first_light
    bypass_plate, plain_plate = make_bypass_plate(axis), make_plate(axis)
    bypassed, plain = first_light.copy(), first_light.copy()
    for plate_count in range(0, 41):
        worst_overlap_gap = max(worst_overlap_gap, abs(axis @ bypassed - 2.0 ** plate_count * first_overlap) / 2.0 ** plate_count)
        bound_holds &= 2.0 ** plate_count * abs(first_overlap) <= np.linalg.norm(bypassed) * (1 + 1e-12)
        if plate_count > 0:
            plain_overlap_change = max(plain_overlap_change, abs(abs(axis @ plain) - abs(first_overlap)))
        bypassed, plain = bypass_plate @ bypassed, plain_plate @ plain
report("overlap after plate_count bypass plates = 2 ^ plate_count * first overlap, plate_count 0 to 40", worst_overlap_gap < 1e-12,
       f"largest gap, relative to 2^plate_count: {worst_overlap_gap:.1e}")
report("the length is at least 2 ^ plate_count * |first overlap|", bound_holds)
report("control: plates without the bypass leave the overlap unchanged, so the doubling is the bypass", plain_overlap_change < 1e-12,
       f"largest change of the overlap {plain_overlap_change:.1e}")

[PASS] overlap after plate_count bypass plates = 2 ^ plate_count * first overlap, plate_count 0 to 40  (largest gap, relative to 2^plate_count: 1.6e-14)
[PASS] the length is at least 2 ^ plate_count * |first overlap|
[PASS] control: plates without the bypass leave the overlap unchanged, so the doubling is the bypass  (largest change of the overlap 2.8e-14)


True

## Claim 5: the fold keeps half (`relu_sq_add_relu_neg_sq`, `integral_relu_sq_of_symmetric`)

First, `relu(x)^2 + relu(-x)^2 = x^2` is an identity of the code, printed as by construction and not counted. Then 2 million standard normal values: the mean of `relu(x)^2` should be half the mean of `x^2`;
the sampling error of that ratio is about `0.001`, so the threshold is `0.005`. The control is a signal that is not symmetric, `x + 1`: the ratio must leave `1/2` by more than `0.05`.

In [6]:
generator = np.random.default_rng(SEED + 3)
grid = np.linspace(-10, 10, 20001)
fold_gap = np.abs(np.maximum(grid, 0) ** 2 + np.maximum(-grid, 0) ** 2 - grid ** 2).max()
print(f"[by construction, cannot fail] relu(x)^2 + relu(-x)^2 = x^2 on 20001 points (largest gap {fold_gap:.1e})")
signal = generator.standard_normal(2_000_000)
kept_fraction = (np.maximum(signal, 0) ** 2).mean() / (signal ** 2).mean()
report("for a symmetric signal the fold keeps half the mean squared size", abs(kept_fraction - 0.5) < 0.005, f"kept fraction {kept_fraction:.4f}")
shifted = signal + 1.0
shifted_fraction = (np.maximum(shifted, 0) ** 2).mean() / (shifted ** 2).mean()
report("control: for the shifted signal x + 1 the fold does not keep half", abs(shifted_fraction - 0.5) > 0.05, f"kept fraction {shifted_fraction:.4f}")

[by construction, cannot fail] relu(x)^2 + relu(-x)^2 = x^2 on 20001 points (largest gap 0.0e+00)


[PASS] for a symmetric signal the fold keeps half the mean squared size  (kept fraction 0.5003)
[PASS] control: for the shifted signal x + 1 the fold does not keep half  (kept fraction 0.9624)


True

## Claim 6: the weight variance `2 / input_count` gives a gain of one (`he_gain`)

First the arithmetic, in exact fractions (by construction, printed and not counted): `input_count * (2 / input_count) * (1/2) = 1` for every `input_count` from 1 to 2000.
Then the network. ReLU layers of width `input_count = 512`, no biases, 50 layers, a batch of 64 standard normal inputs,
weights of variance `weight_variance_constant / 512`. A layer multiplies the mean squared size by `weight_variance_constant / 2`, so the **size** (root mean square over units and batch) has gain
`sqrt(weight_variance_constant / 2)`. Three settings: constant `2 * 0.9^2 = 1.62` (gain 0.9), `2` (He, gain 1), `2 * 1.1^2 = 2.42` (gain 1.1).
Prediction: the slope of `log(size)` against layer is `log(gain)`, which is `-0.105`, `0`, `0.095`. Width 512 gives a sampling error of about `0.02` on the slope, so the threshold is `0.04`.
Also the same for the gradient of a random read-out of the last layer, moving backward from the output (that is a measurement of the backward mask the chapter does not model). The check can fail:
a slope that disagrees with `log(gain)` by more than `0.04` counts against the claim.

In [7]:
exact_gains = {Fraction(input_count) * (Fraction(2) / Fraction(input_count)) * Fraction(1, 2) for input_count in range(1, 2001)}
print(f"[by construction, cannot fail] input_count * (2 / input_count) * (1/2) = 1 for input_count 1 to 2000: {exact_gains == {Fraction(1)}}")

WIDTH, LAYER_COUNT, BATCH_SIZE = 512, 50, 64
GAINS = (0.9, 1.0, 1.1)


def run_relu_network(weight_variance_constant, generator):
    """Forward sizes (rms over units and batch) after each layer, and backward gradient sizes from the output."""
    weights = [generator.standard_normal((WIDTH, WIDTH)) * np.sqrt(weight_variance_constant / WIDTH) for _ in range(LAYER_COUNT)]
    activation = generator.standard_normal((BATCH_SIZE, WIDTH))
    forward_sizes = [np.sqrt((activation ** 2).mean())]
    switches = []
    for weight in weights:
        pre_activation = activation @ weight
        switches.append(pre_activation > 0)
        activation = np.maximum(pre_activation, 0.0)
        forward_sizes.append(np.sqrt((activation ** 2).mean()))
    gradient = np.tile(generator.standard_normal(WIDTH), (BATCH_SIZE, 1))     # a random read-out of the last layer
    backward_sizes = [np.sqrt((gradient ** 2).mean())]
    for weight, switch in zip(reversed(weights), reversed(switches)):
        gradient = (gradient * switch) @ weight.T
        backward_sizes.append(np.sqrt((gradient ** 2).mean()))
    return np.array(forward_sizes), np.array(backward_sizes)


def fit_log_slope(sizes):
    return np.polyfit(np.arange(len(sizes)), np.log(sizes), 1)[0]


generator = np.random.default_rng(SEED + 4)
network_runs = {}
for gain in GAINS:
    weight_variance_constant = 2 * gain ** 2
    forward_sizes, backward_sizes = run_relu_network(weight_variance_constant, generator)
    network_runs[gain] = (forward_sizes, backward_sizes)
    forward_slope, backward_slope = fit_log_slope(forward_sizes), fit_log_slope(backward_sizes)
    report(f"gain {gain}: slope of log(size) against layer is log(gain), forward", abs(forward_slope - np.log(gain)) < 0.04,
           f"measured {forward_slope:+.4f}, predicted {np.log(gain):+.4f}, size after 50 layers {forward_sizes[-1]:.3g} vs gain^50 = {gain ** 50:.3g}")
    report(f"gain {gain}: the same slope for the gradient, backward", abs(backward_slope - np.log(gain)) < 0.04,
           f"measured {backward_slope:+.4f}, predicted {np.log(gain):+.4f}, size after 50 layers {backward_sizes[-1]:.3g}")
wrong_prediction_gap = abs(fit_log_slope(network_runs[0.9][0]) - np.log(1.0))
report("control: the prediction 'gain one' for the variance 1.62 / 512 is wrong by more than the threshold", wrong_prediction_gap > 0.04,
       f"slope {fit_log_slope(network_runs[0.9][0]):+.4f} against 0 for gain one")

[by construction, cannot fail] input_count * (2 / input_count) * (1/2) = 1 for input_count 1 to 2000: True


[PASS] gain 0.9: slope of log(size) against layer is log(gain), forward  (measured -0.1102, predicted -0.1054, size after 50 layers 0.0034 vs gain^50 = 0.00515)
[PASS] gain 0.9: the same slope for the gradient, backward  (measured -0.1096, predicted -0.1054, size after 50 layers 0.00403)


[PASS] gain 1.0: slope of log(size) against layer is log(gain), forward  (measured -0.0073, predicted +0.0000, size after 50 layers 0.833 vs gain^50 = 1)
[PASS] gain 1.0: the same slope for the gradient, backward  (measured -0.0061, predicted +0.0000, size after 50 layers 0.78)


[PASS] gain 1.1: slope of log(size) against layer is log(gain), forward  (measured +0.0957, predicted +0.0953, size after 50 layers 103 vs gain^50 = 117)
[PASS] gain 1.1: the same slope for the gradient, backward  (measured +0.0941, predicted +0.0953, size after 50 layers 106)
[PASS] control: the prediction 'gain one' for the variance 1.62 / 512 is wrong by more than the threshold  (slope -0.1102 against 0 for gain one)


True

## Claim 7: a stack of sigmoids has slope at most `(1/4) ^ layer_count` (`lipschitzWith_sigmoid`, `lipschitzWith_iterate_sigmoid`)

The slope of the stack is measured by central finite differences of the stack itself (not by the chain-rule formula), on 40001 points in `[-12, 12]`, for `layer_count` from 1 to 10.
Prediction: the largest slope is at most `(1/4) ^ layer_count` for every depth, and for one layer it equals `1/4` (reached at 0, to within `1e-6` from the finite-difference step). The control is a stack of `tanh` layers, whose slope at 0 is 1:
the bound `(1/4) ^ layer_count` must fail for it.

In [8]:
inputs = np.linspace(-12, 12, 40001)


def measure_stack_slope(activation, layer_count):
    """Largest slope of the stack, by central differences of the stack's own values."""
    value = inputs.copy()
    for layer in range(layer_count):
        value = activation(value)
    return np.abs(np.gradient(value, inputs)).max()


def sigmoid(value):
    return 1 / (1 + np.exp(-value))


SIGMOID_LAYERS = np.arange(1, 11)
sigmoid_slopes = np.array([measure_stack_slope(sigmoid, count) for count in SIGMOID_LAYERS])
tanh_slopes = np.array([measure_stack_slope(np.tanh, count) for count in SIGMOID_LAYERS])
slope_limits = 0.25 ** SIGMOID_LAYERS
report("the largest slope of a stack of sigmoids is at most (1/4) ^ layer_count, layer_count 1 to 10",
       np.all(sigmoid_slopes <= slope_limits * (1 + 1e-6)), f"largest ratio to the limit {np.max(sigmoid_slopes / slope_limits):.4f}")
report("for one layer the limit 1/4 is reached", abs(sigmoid_slopes[0] - 0.25) < 1e-6, f"measured {sigmoid_slopes[0]:.9f}")
print(f"ten sigmoid layers: largest slope {sigmoid_slopes[-1]:.3e}, limit {slope_limits[-1]:.3e}, ratio {sigmoid_slopes[-1] / slope_limits[-1]:.3f}")
report("control: a stack of tanh layers breaks the bound (slope 1 at 0)", np.all(tanh_slopes > slope_limits[0] * 3.9),
       f"tanh stack slope after 10 layers {tanh_slopes[-1]:.3f}, bound {slope_limits[-1]:.1e}")

[PASS] the largest slope of a stack of sigmoids is at most (1/4) ^ layer_count, layer_count 1 to 10  (largest ratio to the limit 1.0000)
[PASS] for one layer the limit 1/4 is reached  (measured 0.249999993)
ten sigmoid layers: largest slope 3.901e-07, limit 9.537e-07, ratio 0.409
[PASS] control: a stack of tanh layers breaks the bound (slope 1 at 0)  (tanh stack slope after 10 layers 1.000, bound 9.5e-07)


True

## Figures

The first figure shows the size of the signal and of the gradient against depth in the ReLU network of claim 6, for gains 0.9, 1 and 1.1, with the predicted `gain ^ depth` and
the proved bound. The second shows the sigmoid stack slope against the `(1/4) ^ layer_count` limit, with the `tanh` control.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4), sharey=True)
LABELS = {0.9: "gain 0.9 (variance 1.62 / width)", 1.0: "gain 1 (He: variance 2 / width)", 1.1: "gain 1.1 (variance 2.42 / width)"}
STYLES = {0.9: ("o", 0.45), 1.0: ("s", 0.75), 1.1: ("^", 1.0)}
layers = np.arange(LAYER_COUNT + 1)
for axis, title, which in zip(axes, ("Signal size going forward", "Gradient size going backward"), (0, 1)):
    for gain in GAINS:
        marker, strength = STYLES[gain]
        axis.plot(layers, network_runs[gain][which], marker=marker, markevery=5, markersize=3.5, color=MODEL, alpha=strength,
                  label=f"measured, {LABELS[gain]}")
    for gain in GAINS:
        axis.plot(layers, gain ** layers, color="black", linestyle="--", linewidth=0.9,
                  label="predicted, gain ^ layers" if gain == GAINS[0] else None)
    axis.plot(layers, np.exp(-layers * (1 - 0.9)), color=BOUND, linestyle=":", linewidth=1.5,
              label="proved bound, exp(-layers x 0.1)")
    axis.set_yscale("log")
    axis.set_xlabel("layers passed" if which == 0 else "layers passed, counted back from the output")
    axis.set_title(title)
axes[0].set_ylabel("size (root mean square)")
fig.legend(*axes[0].get_legend_handles_labels(), fontsize=6, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.17))
fig.suptitle("A gain below one dies, above one blows up, one survives (ReLU network, width 512)", y=1.02, fontsize=9)
save_figure(fig, "30-gain-against-depth")
plt.close(fig)

fig, axis = plt.subplots(figsize=(5.5, 3.3))
axis.plot(SIGMOID_LAYERS, sigmoid_slopes, marker="o", color=MODEL, label="measured, sigmoid stack")
axis.plot(SIGMOID_LAYERS, slope_limits, color=BOUND, linestyle="--", label="proved limit (1/4) ^ layers")
axis.plot(SIGMOID_LAYERS, tanh_slopes, marker="s", color=BEFORE, label="control: tanh stack, limit does not apply")
axis.set_yscale("log")
axis.set_xlabel("layers in the stack")
axis.set_ylabel("largest slope of the stack")
axis.set_title("A stack of sigmoids loses at least a factor 4 of slope per layer")
axis.legend(fontsize=7)
save_figure(fig, "30-sigmoid-stack-slope")
plt.close(fig)

wrote book/figures/30-gain-against-depth.png


wrote book/figures/30-sigmoid-stack-slope.png
